# Soul Path Classification Analysis

**Purpose**: Develop discriminant models to classify NDErs into soul path categories

**Research Question**: Can we statistically distinguish between:
1. **Volunteer Souls** — Pre-birth mission-focused individuals
2. **Standard Path** — Typical continuation trajectory
3. **Restorative Path** — Exceptional re-embodiment cases

## Theoretical Framework

The **Threefold Path Cosmology** posits three distinct trajectories:

| Path | Description | NDE Signature |
|------|-------------|---------------|
| **Affirmation** | Union with Divine (Heaven) | Welcoming light, loving beings |
| **Negation** | Separation from Divine (Hell) | Distressing NDEs, void/darkness |
| **Restorative** | Re-embodiment for remediation | Mission focus, pre-birth memories |

The **Volunteer Soul** hypothesis suggests a subset of incarnated souls:
- Chose embodiment for specific mission
- May have pre-birth memories
- Experience distinctive NDE phenomenology

## Data Sources

- NDERF: 5,646 structured experiences
- IANDS: 1,093 structured experiences  
- Total: 6,739 experiences

## Schema Improvements Applied

| OLD Schema Problem | NEW Schema Solution | Classification Opportunity |
|-------------------|---------------------|---------------------------|
| Single `chose_current_life` | `chose_parents` + `chose_mission` + `chose_life_circumstances` | Independent binary features |
| Conflated mission/purpose | `mission_given` vs `purpose_revealed` | Distinguish pre-birth vs during-NDE |
| Single return field | `return_agency` + `return_willingness` | Pattern combinations as features |

---

**Notebook**: `03_soul_path_classification.ipynb`  
**Schema Version**: 2026-01-06  
**Source Notebooks**: `volunteer_discriminant_analysis.ipynb`, `threefold_path_validation.ipynb`

## Setup: Load Libraries and Data

In [ ]:
# Standard libraries
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

# Data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr

# Machine Learning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print("Libraries loaded successfully")

In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

def load_all_experiences():
    """Load all structured NDE experiences from both NDERF and IANDS."""
    experiences = []
    data_dir = Path('../structured')
    
    for json_file in data_dir.glob('*.json'):
        with open(json_file, 'r', encoding='utf-8') as f:
            exp = json.load(f)
            exp['_source'] = 'NDERF' if 'nderf' in json_file.stem.lower() else 'IANDS'
            exp['_file'] = json_file.stem
            experiences.append(exp)
    
    return experiences

# Load data
all_experiences = load_all_experiences()
print(f"Total experiences loaded: {len(all_experiences)}")

# Source breakdown
sources = Counter(exp['_source'] for exp in all_experiences)
for source, count in sources.items():
    print(f"  {source}: {count}")

In [ ]:
# =============================================================================
# EXTRACT CLASSIFICATION FEATURES
# =============================================================================

def extract_classification_features(exp):
    """Extract features relevant to soul path classification."""
    return {
        # Identifiers
        'id': exp.get('_file'),
        'source': exp.get('_source'),
        
        # Pre-birth choice indicators (NEW split fields)
        'chose_parents': exp.get('chose_parents'),  # Binary
        'chose_mission': exp.get('chose_mission'),  # Binary
        'chose_life_circumstances': exp.get('chose_life_circumstances'),  # Binary
        
        # Mission/purpose indicators
        'mission_given': exp.get('mission_given'),  # During NDE
        'purpose_revealed': exp.get('purpose_revealed'),  # During NDE
        'sense_of_mission': exp.get('sense_of_mission'),  # After NDE
        
        # Return indicators (NEW split fields)
        'return_agency': exp.get('return_agency'),
        'return_willingness': exp.get('return_willingness'),
        'return_reasons': exp.get('return_reasons', []),
        
        # Past life indicators
        'past_lives_shown': exp.get('past_lives_shown'),
        'reincarnation_mentioned': exp.get('reincarnation_mentioned'),
        
        # Being encounters
        'being_encounter': exp.get('being_encounter'),
        'light_being_present': exp.get('light_being_present'),
        'guidance_received': exp.get('guidance_received'),
        
        # Experience quality
        'experience_valence': exp.get('experience_valence'),  # Positive/Negative/Mixed
        'life_review_occurred': exp.get('life_review_occurred'),
        
        # Transformation (NEW before/after)
        'death_fear_before': exp.get('death_fear_before'),
        'death_fear_after': exp.get('death_fear_after'),
        'spirituality_before': exp.get('spirituality_before'),
        'spirituality_after': exp.get('spirituality_after'),
        
        # Demographics
        'gender': exp.get('gender'),
        'age_at_experience': exp.get('age_at_experience'),
        'religious_background': exp.get('religious_background'),
    }

# Extract features
feature_data = [extract_classification_features(exp) for exp in all_experiences]
feature_df = pd.DataFrame(feature_data)

print(f"Extracted {len(feature_df)} records with {len(feature_df.columns)} features")
print(f"\nKey volunteer indicators:")
for col in ['chose_parents', 'chose_mission', 'chose_life_circumstances', 'mission_given', 'sense_of_mission']:
    non_null = feature_df[col].notna().sum()
    yes_count = (feature_df[col] == 'YES').sum() if non_null > 0 else 0
    print(f"  {col}: {yes_count} YES / {non_null} with data ({yes_count/non_null*100:.1f}% if data)" if non_null > 0 else f"  {col}: no data")

---

# PART I: Volunteer Soul Identification

**Objective**: Identify potential "Volunteer Souls" based on pre-birth choice indicators.

**Hypothesis**: A subset of NDErs will show distinctive patterns suggesting pre-incarnation mission awareness.

In [ ]:
# =============================================================================
# PART I: VOLUNTEER SOUL INDICATOR ANALYSIS (NEW split fields)
# =============================================================================

print("=" * 70)
print("VOLUNTEER SOUL INDICATORS (NEW Schema: Split Choice Fields)")
print("=" * 70)
print("✓ NEW: Separate fields for chose_parents, chose_mission, chose_life_circumstances")
print()

# 1. Individual indicator analysis
print("1. Pre-Birth Choice Indicators:")
print("-" * 50)

choice_fields = ['chose_parents', 'chose_mission', 'chose_life_circumstances']

choice_stats = []
for field in choice_fields:
    counts = feature_df[field].value_counts()
    total = counts.sum()
    yes_count = counts.get('YES', 0)
    no_count = counts.get('NO', 0)
    
    print(f"\n   {field}:")
    print(f"      YES: {yes_count} ({yes_count/total*100:.1f}%)" if total > 0 else f"      No data")
    print(f"      NO: {no_count} ({no_count/total*100:.1f}%)" if total > 0 else "")
    
    choice_stats.append({
        'field': field,
        'yes': yes_count,
        'no': no_count,
        'total': total,
        'yes_rate': yes_count/total*100 if total > 0 else 0
    })

# 2. Composite volunteer score
print("\n\n2. Composite Volunteer Score:")
print("-" * 50)
print("Creating score: +1 for each YES in choice fields")

def compute_volunteer_score(row):
    """Compute volunteer soul score from choice indicators."""
    score = 0
    for field in choice_fields:
        if row.get(field) == 'YES':
            score += 1
    return score

feature_df['volunteer_score'] = feature_df.apply(compute_volunteer_score, axis=1)

# Score distribution
score_counts = feature_df['volunteer_score'].value_counts().sort_index()
print("\nVolunteer Score Distribution:")
for score, count in score_counts.items():
    print(f"   Score {score}: {count} ({count/len(feature_df)*100:.1f}%)")

# Define volunteer threshold
VOLUNTEER_THRESHOLD = 2  # At least 2 out of 3 choice indicators

volunteer_candidates = feature_df[feature_df['volunteer_score'] >= VOLUNTEER_THRESHOLD]
print(f"\n✓ Volunteer Candidates (score >= {VOLUNTEER_THRESHOLD}): {len(volunteer_candidates)} ({len(volunteer_candidates)/len(feature_df)*100:.1f}%)")

In [ ]:
# =============================================================================
# MISSION INDICATORS (NEW capability: distinguish pre-birth vs during-NDE)
# =============================================================================

print("=" * 70)
print("MISSION & PURPOSE INDICATORS")
print("=" * 70)
print("✓ NEW: Distinguish mission_given (during NDE) from sense_of_mission (after)")
print()

# 1. Mission during NDE
print("1. Mission/Purpose During NDE:")
print("-" * 50)

mission_counts = feature_df['mission_given'].value_counts()
purpose_counts = feature_df['purpose_revealed'].value_counts()

print(f"   Mission given during NDE: {mission_counts.get('YES', 0)} YES / {mission_counts.sum()} total")
print(f"   Purpose revealed during NDE: {purpose_counts.get('YES', 0)} YES / {purpose_counts.sum()} total")

# 2. Post-NDE mission sense
print("\n2. Sense of Mission After NDE:")
print("-" * 50)

sense_counts = feature_df['sense_of_mission'].value_counts()
print(f"   Sense of mission after NDE: {sense_counts.get('YES', 0)} YES / {sense_counts.sum()} total")

# 3. Cross-tabulation: Pre-birth choice × Mission given
print("\n3. Pre-Birth Choice × Mission Given:")
print("-" * 50)
print("(Do those who 'chose' their life also receive mission during NDE?)")

valid_both = feature_df[(feature_df['chose_mission'].isin(['YES', 'NO'])) & 
                        (feature_df['mission_given'].isin(['YES', 'NO']))]
if len(valid_both) > 0:
    crosstab = pd.crosstab(valid_both['chose_mission'], valid_both['mission_given'], margins=True)
    print(crosstab.to_string())
    
    # Chi-square test
    chi2, p_value, dof, expected = chi2_contingency(crosstab.iloc[:-1, :-1])
    print(f"\nChi-square: {chi2:.2f}, p-value: {p_value:.4f}")
    print(f"{'✓ SIGNIFICANT' if p_value < 0.05 else '○ Not significant'}: Pre-birth choice {'IS' if p_value < 0.05 else 'is NOT'} associated with mission during NDE")
else:
    print("Insufficient data for cross-tabulation")

---
## Part II: Trauma Marker Analysis (Restorative Indicators)

The **Restorative Path** is indicated by **cyclic reincarnation markers** - memories of previous deaths and past lives that suggest unresolved trauma requiring healing:

| Indicator | Meaning |
|-----------|---------|
| `death_memory` | Memory of dying in a previous life |
| `past_life_memory` | Recall of previous incarnation details |
| `intermission_memory` | Memory of the between-life state |

**Hypothesis**: High trauma markers WITHOUT volunteer indicators → Restorative Path

In [ ]:
# =============================================================================
# TRAUMA MARKER DISTRIBUTIONS
# =============================================================================
print("="*80)
print("TRAUMA MARKER DISTRIBUTIONS")
print("="*80)

# Check what cosmic knowledge fields are available
cosmic_fields = [col for col in df.columns if 'death_memory' in col or 'past_life' in col or 'intermission' in col]
print(f"\nAvailable trauma marker fields: {cosmic_fields}")

# If direct fields available, use them; otherwise check cosmic_knowledge dict
if 'cosmic_knowledge' in df.columns:
    # Extract from nested dict
    def get_trauma_markers(cosmic_knowledge):
        if isinstance(cosmic_knowledge, dict):
            return {
                'death_memory': cosmic_knowledge.get('death_memory', 'not_mentioned'),
                'past_life_memory': cosmic_knowledge.get('past_life_memory', 'not_mentioned'),
                'intermission_memory': cosmic_knowledge.get('intermission_memory', 'not_mentioned')
            }
        return {'death_memory': 'not_mentioned', 'past_life_memory': 'not_mentioned', 'intermission_memory': 'not_mentioned'}
    
    trauma_markers = df['cosmic_knowledge'].apply(get_trauma_markers).apply(pd.Series)
    for col in trauma_markers.columns:
        df[col] = trauma_markers[col]

# Distribution analysis
for marker in ['death_memory', 'past_life_memory', 'intermission_memory']:
    if marker in df.columns:
        print(f"\n{marker.upper().replace('_', ' ')}:")
        print(df[marker].value_counts())

# Create trauma indicator flags
df['has_death_memory'] = df.get('death_memory', pd.Series(['not_mentioned']*len(df))).isin(['yes_explicit', 'implied', 'vague'])
df['has_past_life_memory'] = df.get('past_life_memory', pd.Series(['not_mentioned']*len(df))).isin(['yes_explicit', 'implied', 'vague'])
df['has_trauma_markers'] = df['has_death_memory'] | df['has_past_life_memory']

print(f"\n\nTRAUMA MARKER SUMMARY:")
print(f"Has death memory: {df['has_death_memory'].sum():,} ({df['has_death_memory'].mean()*100:.2f}%)")
print(f"Has past life memory: {df['has_past_life_memory'].sum():,} ({df['has_past_life_memory'].mean()*100:.2f}%)")
print(f"Has EITHER (Restorative indicator): {df['has_trauma_markers'].sum():,} ({df['has_trauma_markers'].mean()*100:.2f}%)")

In [ ]:
# =============================================================================
# TRAUMA × VOLUNTEER CROSS-TABULATION
# =============================================================================
print("="*80)
print("TRAUMA MARKERS × VOLUNTEER INDICATORS")
print("="*80)

# Create volunteer indicator if not already present
if 'has_volunteer_markers' not in df.columns:
    df['has_volunteer_markers'] = (
        df.get('volunteer_language', pd.Series(['not_mentioned']*len(df))).isin(['yes_explicit', 'implied']) |
        (df.get('return_reason', pd.Series(['not_mentioned']*len(df))) == 'earthly_mission') |
        (df.get('mission_commissioned', pd.Series(['no']*len(df))).isin(['yes_explicit', 'implied']))
    )

# Cross-tabulation
print("\nCross-tabulation: Trauma Markers × Volunteer Markers")
cross_tab = pd.crosstab(
    df['has_trauma_markers'].map({True: 'Has Trauma', False: 'No Trauma'}),
    df['has_volunteer_markers'].map({True: 'Has Volunteer', False: 'No Volunteer'}),
    margins=True
)
print(cross_tab)

# Chi-square test
from scipy import stats
contingency = pd.crosstab(df['has_trauma_markers'], df['has_volunteer_markers'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test: χ²={chi2:.2f}, p={p:.2e}, df={dof}")

# Preliminary soul path classification
print("\n\nPRELIMINARY SOUL PATH CLASSIFICATION:")
df['preliminary_soul_path'] = 'Indeterminate'
df.loc[df['has_trauma_markers'] & ~df['has_volunteer_markers'], 'preliminary_soul_path'] = 'Restorative'
df.loc[df['has_volunteer_markers'] & ~df['has_trauma_markers'], 'preliminary_soul_path'] = 'Volunteer'
df.loc[df['has_volunteer_markers'] & df['has_trauma_markers'], 'preliminary_soul_path'] = 'Hybrid'

print(df['preliminary_soul_path'].value_counts())
print("\nPercentages:")
print((df['preliminary_soul_path'].value_counts(normalize=True) * 100).round(2))

---
## Part III: Ohkado Pattern Analysis (Pre-Birth Awareness)

**Ohkado Cases** show pre-birth awareness WITHOUT trauma markers - potential first-incarnation volunteers or souls with intact pre-mortal memory:

The **5 Pre-Birth Indicators** (from Ohkado research):
1. `premortal_existence_info` - Information about existence before this life
2. `pre_birth_realm_description` - Description of the realm before incarnation  
3. `incarnation_choice` - Evidence of choosing to incarnate (NEW: now split into chose_parents, chose_mission, chose_life_circumstances)
4. `home_identification` - Identifies spiritual realm as "home"
5. `identity_pre_body` - Sense of identity existing before physical body

**Perfect 5-Indicator Cases** = All 5 present → Strong Volunteer/Ohkado profile

In [ ]:
# =============================================================================
# OHKADO PATTERN: PRE-BIRTH INDICATOR COUNT
# =============================================================================
print("="*80)
print("OHKADO PATTERN ANALYSIS: PRE-BIRTH INDICATORS")
print("="*80)

# Extract pre-birth fields from cosmic_knowledge if needed
prebirth_fields = ['premortal_existence_info', 'pre_birth_realm_description', 
                   'home_identification', 'identity_pre_body']

# Check for new split incarnation_choice fields
choice_fields = ['chose_parents', 'chose_mission', 'chose_life_circumstances']

for field in prebirth_fields:
    if field not in df.columns and 'cosmic_knowledge' in df.columns:
        df[field] = df['cosmic_knowledge'].apply(lambda x: x.get(field, 'not_mentioned') if isinstance(x, dict) else 'not_mentioned')

# Calculate indicator count using NEW schema fields
def count_prebirth_indicators(row):
    count = 0
    # Traditional indicators
    for field in prebirth_fields:
        if row.get(field, 'not_mentioned') in ['yes_explicit', 'implied', 'spiritual_realm']:
            count += 1
    # NEW split choice fields - any choice = 1 indicator point
    has_any_choice = any(
        row.get(f, 'not_mentioned') in ['yes_explicit', 'implied', True] 
        for f in choice_fields
    )
    if has_any_choice:
        count += 1
    return count

df['prebirth_indicator_count'] = df.apply(count_prebirth_indicators, axis=1)

print("\nPre-birth Indicator Count Distribution:")
print(df['prebirth_indicator_count'].value_counts().sort_index())

# Perfect 5-indicator cases
perfect_5 = df[df['prebirth_indicator_count'] == 5]
print(f"\n\n🌟 PERFECT 5-INDICATOR CASES (Strong Ohkado/Volunteer): {len(perfect_5)} ({len(perfect_5)/len(df)*100:.2f}%)")

# High indicator cases (4+)
high_indicator = df[df['prebirth_indicator_count'] >= 4]
print(f"📊 HIGH INDICATOR CASES (4+): {len(high_indicator)} ({len(high_indicator)/len(df)*100:.2f}%)")

In [ ]:
# =============================================================================
# OHKADO CLASSIFICATION: Pre-birth Awareness WITHOUT Trauma
# =============================================================================
print("="*80)
print("OHKADO CLASSIFICATION: PRE-BIRTH WITHOUT TRAUMA")
print("="*80)

# Has pre-birth awareness (any indicator)
df['has_prebirth_awareness'] = df['prebirth_indicator_count'] > 0

# Ohkado pattern: Pre-birth awareness WITHOUT trauma markers
# These may represent first-incarnation volunteers with intact pre-mortal memory
df['is_ohkado'] = df['has_prebirth_awareness'] & ~df['has_trauma_markers'] & ~df['has_volunteer_markers']

print(f"\nCases with Pre-birth Awareness: {df['has_prebirth_awareness'].sum():,} ({df['has_prebirth_awareness'].mean()*100:.1f}%)")
print(f"Cases fitting Ohkado Pattern: {df['is_ohkado'].sum():,} ({df['is_ohkado'].mean()*100:.1f}%)")

# Update preliminary classification to include Ohkado
df.loc[df['is_ohkado'], 'preliminary_soul_path'] = 'Ohkado'

print("\n\nUPDATED SOUL PATH CLASSIFICATION (with Ohkado):")
print(df['preliminary_soul_path'].value_counts())

# Cross-tab: Ohkado indicator count vs trauma markers
print("\n\nPre-birth Indicator Count × Trauma Markers:")
cross = pd.crosstab(df['prebirth_indicator_count'], df['has_trauma_markers'].map({True: 'Has Trauma', False: 'No Trauma'}))
print(cross)

In [ ]:
# =============================================================================
# STRONG OHKADO TIERS AND CASE INSPECTION
# =============================================================================
print("="*80)
print("OHKADO CANDIDATE TIERS")
print("="*80)

# Define no_trauma_markers if not already present
no_trauma_markers = ~df['has_trauma_markers']

# Strong candidates: 2+ indicators AND no trauma markers
strong_ohkado = (df['prebirth_indicator_count'] >= 2) & no_trauma_markers
df['strong_ohkado'] = strong_ohkado

# Very strong candidates: 3+ indicators
very_strong_ohkado = (df['prebirth_indicator_count'] >= 3) & no_trauma_markers
df['very_strong_ohkado'] = very_strong_ohkado

# Perfect candidates: All 5 indicators
perfect_ohkado = (df['prebirth_indicator_count'] == 5) & no_trauma_markers
df['perfect_ohkado'] = perfect_ohkado

print(f"\nOhkado Tiers (without trauma markers):")
print(f"  Any (1+ indicators): {df['is_ohkado'].sum():,} ({df['is_ohkado'].mean()*100:.2f}%)")
print(f"  Strong (2+ indicators): {strong_ohkado.sum():,} ({strong_ohkado.mean()*100:.2f}%)")
print(f"  Very Strong (3+ indicators): {very_strong_ohkado.sum():,} ({very_strong_ohkado.mean()*100:.2f}%)")
print(f"  Perfect (5 indicators): {perfect_ohkado.sum():,} ({perfect_ohkado.mean()*100:.2f}%)")

print(f"\n\nIndicator Count Distribution (Ohkado pattern only):")
ohkado_counts = df[df['is_ohkado']]['prebirth_indicator_count'].value_counts().sort_index()
print(ohkado_counts)

# Helper function for case display
import textwrap

def display_ohkado_case(row, max_content=2000):
    """Display Ohkado candidate case details."""
    print("="*80)
    print(f"TITLE: {row.get('title', 'Unknown')}")
    print(f"Dataset: {row.get('dataset', 'Unknown')} | Source: {row.get('source_file', 'Unknown')}")
    print("-"*80)
    
    print(f"\nPRE-BIRTH INDICATORS (Count: {row.get('prebirth_indicator_count', 0)}/5):")
    for field in ['premortal_existence_info', 'pre_birth_realm_description', 'home_identification', 'identity_pre_body']:
        val = row.get(field, 'not_mentioned')
        marker = "✓" if val in ['yes_explicit', 'implied', 'spiritual_realm'] else "✗"
        print(f"  {marker} {field}: {val}")
    # Check split choice fields
    for field in ['chose_parents', 'chose_mission', 'chose_life_circumstances']:
        val = row.get(field, 'not_mentioned')
        if val in ['yes_explicit', 'implied', True]:
            print(f"  ✓ {field}: {val}")
    
    print(f"\nTRAUMA MARKERS (should be absent):")
    print(f"  death_memory: {row.get('death_memory', 'not_mentioned')}")
    print(f"  past_life_memory: {row.get('past_life_memory', 'not_mentioned')}")
    
    print(f"\nSOUL PATH: {row.get('preliminary_soul_path', 'Unknown')}")
    print()

---

# PART II: Experience Valence Classification (Threefold Path)

**Objective**: Classify experiences by valence as proxy for path trajectory.

**Framework**:
- **Positive NDEs** → Path of Affirmation (majority)
- **Distressing NDEs** → Path of Negation (minority)  
- **Mixed NDEs** → Transitional or Restorative

In [ ]:
# =============================================================================
# PART II: EXPERIENCE VALENCE ANALYSIS
# =============================================================================

print("=" * 70)
print("EXPERIENCE VALENCE CLASSIFICATION")
print("=" * 70)

# 1. Valence distribution
print("1. Experience Valence Distribution:")
print("-" * 50)

valence_counts = feature_df['experience_valence'].value_counts()
total_valence = valence_counts.sum()

for valence, count in valence_counts.items():
    print(f"   {valence}: {count} ({count/total_valence*100:.1f}%)")

# 2. Map to threefold path
print("\n\n2. Mapping to Threefold Path:")
print("-" * 50)

def map_to_path(valence):
    """Map experience valence to threefold path."""
    if valence in ['POSITIVE', 'TRANSCENDENT', 'BLISSFUL']:
        return 'AFFIRMATION'
    elif valence in ['NEGATIVE', 'DISTRESSING', 'HELLISH', 'VOID']:
        return 'NEGATION'
    elif valence in ['MIXED', 'NEUTRAL']:
        return 'TRANSITIONAL'
    else:
        return 'UNKNOWN'

feature_df['path_estimate'] = feature_df['experience_valence'].apply(map_to_path)

path_counts = feature_df['path_estimate'].value_counts()
for path, count in path_counts.items():
    print(f"   {path}: {count} ({count/len(feature_df)*100:.1f}%)")

# 3. Distressing NDE analysis
print("\n\n3. Distressing NDE Analysis (Path of Negation):")
print("-" * 50)

distressing = feature_df[feature_df['path_estimate'] == 'NEGATION']
print(f"Total distressing NDEs: {len(distressing)}")

if len(distressing) > 0:
    # What are their characteristics?
    print("\nCharacteristics of distressing NDErs:")
    
    # Religious background
    dist_religion = distressing['religious_background'].value_counts()
    print(f"\n   Top religious backgrounds:")
    for bg, count in dist_religion.head(5).items():
        print(f"      {bg}: {count}")
    
    # Return patterns
    dist_agency = distressing['return_agency'].value_counts()
    print(f"\n   Return agency:")
    for agency, count in dist_agency.items():
        print(f"      {agency}: {count}")

---

# PART III: Discriminant Analysis - Volunteer vs Standard

**Objective**: Build a discriminant model to distinguish Volunteer Souls from Standard Path.

**Method**: Linear Discriminant Analysis and Random Forest classification.

In [ ]:
# =============================================================================
# PART III: PREPARE ML DATASET
# =============================================================================

print("=" * 70)
print("PREPARING ML DATASET FOR DISCRIMINANT ANALYSIS")
print("=" * 70)

# Create binary target: Volunteer (score >= 2) vs Standard (score < 2)
feature_df['is_volunteer'] = (feature_df['volunteer_score'] >= VOLUNTEER_THRESHOLD).astype(int)

print(f"Target variable distribution:")
print(f"   Volunteer (1): {feature_df['is_volunteer'].sum()}")
print(f"   Standard (0): {(feature_df['is_volunteer'] == 0).sum()}")

# Define feature columns for ML
binary_features = [
    'mission_given', 'purpose_revealed', 'sense_of_mission',
    'past_lives_shown', 'reincarnation_mentioned',
    'being_encounter', 'light_being_present', 'guidance_received',
    'life_review_occurred'
]

categorical_features = [
    'return_agency', 'return_willingness', 'experience_valence',
    'gender', 'religious_background'
]

# Convert to numeric
print("\n\nEncoding features...")

ml_df = feature_df.copy()

# Binary features: YES=1, NO=0, others=NaN
for col in binary_features:
    ml_df[f'{col}_num'] = ml_df[col].apply(lambda x: 1 if x == 'YES' else (0 if x == 'NO' else np.nan))

# Categorical features: Label encoding
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    valid_mask = ml_df[col].notna()
    if valid_mask.sum() > 0:
        ml_df.loc[valid_mask, f'{col}_num'] = le.fit_transform(ml_df.loc[valid_mask, col])
        label_encoders[col] = le

# Numeric features
ml_df['age_num'] = pd.to_numeric(ml_df['age_at_experience'], errors='coerce')

# Build feature matrix
feature_cols = [f'{col}_num' for col in binary_features] + \
               [f'{col}_num' for col in categorical_features] + \
               ['age_num']

# Filter to rows with sufficient data
valid_rows = ml_df[feature_cols + ['is_volunteer']].dropna()
print(f"\nRows with complete feature data: {len(valid_rows)} ({len(valid_rows)/len(ml_df)*100:.1f}%)")

if len(valid_rows) < 50:
    print("\n⚠️ WARNING: Insufficient data for reliable ML analysis")
    print("   Will proceed with available data but results should be interpreted cautiously")

In [ ]:
# =============================================================================
# DISCRIMINANT ANALYSIS
# =============================================================================

print("=" * 70)
print("DISCRIMINANT ANALYSIS: Volunteer vs Standard")
print("=" * 70)

if len(valid_rows) >= 30:
    X = valid_rows[feature_cols].values
    y = valid_rows['is_volunteer'].values
    
    # Handle any remaining NaN (shouldn't be any, but safety check)
    X = np.nan_to_num(X, nan=0)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Training set: {len(X_train)} (Volunteer: {sum(y_train)})")
    print(f"Test set: {len(X_test)} (Volunteer: {sum(y_test)})")
    
    # 1. Linear Discriminant Analysis
    print("\n\n1. Linear Discriminant Analysis:")
    print("-" * 50)
    
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train, y_train)
    lda_pred = lda.predict(X_test)
    lda_acc = accuracy_score(y_test, lda_pred)
    
    print(f"   Accuracy: {lda_acc:.1%}")
    print(f"\n   Classification Report:")
    print(classification_report(y_test, lda_pred, target_names=['Standard', 'Volunteer']))
    
    # 2. Random Forest
    print("\n\n2. Random Forest Classifier:")
    print("-" * 50)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    
    print(f"   Accuracy: {rf_acc:.1%}")
    
    # Cross-validation
    cv_scores = cross_val_score(rf, X, y, cv=5)
    print(f"   Cross-val accuracy: {cv_scores.mean():.1%} (±{cv_scores.std():.1%})")
    
    # 3. Feature Importance
    print("\n\n3. Feature Importance (Random Forest):")
    print("-" * 50)
    
    importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    for _, row in importance_df.head(10).iterrows():
        print(f"   {row['Feature'].replace('_num', '')}: {row['Importance']:.3f}")
else:
    print("Insufficient data for discriminant analysis")
    lda_acc = None
    rf_acc = None
    importance_df = None

---

# PART IV: Return Pattern Analysis (Agency × Willingness)

**Objective**: Analyze the NEW Agency × Willingness combinations as classification features.

**Hypothesis**: Different soul paths have distinctive return patterns.

In [ ]:
# =============================================================================
# PART IV: RETURN PATTERN ANALYSIS
# =============================================================================

print("=" * 70)
print("RETURN PATTERN ANALYSIS BY SOUL PATH")
print("=" * 70)
print("✓ NEW: Agency × Willingness combinations as classification features")
print()

# Create combined return pattern
feature_df['return_pattern'] = feature_df['return_agency'].fillna('UNK') + '_' + feature_df['return_willingness'].fillna('UNK')

# 1. Return patterns by volunteer status
print("1. Return Patterns by Volunteer Status:")
print("-" * 50)

vol_patterns = feature_df[feature_df['is_volunteer'] == 1]['return_pattern'].value_counts()
std_patterns = feature_df[feature_df['is_volunteer'] == 0]['return_pattern'].value_counts()

print("\nVolunteer Soul Return Patterns:")
for pattern, count in vol_patterns.head(6).items():
    total_vol = len(feature_df[feature_df['is_volunteer'] == 1])
    print(f"   {pattern}: {count} ({count/total_vol*100:.1f}%)")

print("\nStandard Path Return Patterns:")
for pattern, count in std_patterns.head(6).items():
    total_std = len(feature_df[feature_df['is_volunteer'] == 0])
    print(f"   {pattern}: {count} ({count/total_std*100:.1f}%)")

# 2. Key pattern analysis
print("\n\n2. Key Pattern Analysis:")
print("-" * 50)

# SELF_WILLING: Chose and wanted to return (mission completion?)
# EXTERNAL_RELUCTANT: Sent back despite wanting to stay (interrupted journey)
# SELF_RELUCTANT: Chose to return despite not wanting (duty)

patterns_of_interest = ['SELF_WILLING', 'SELF_RELUCTANT', 'EXTERNAL_WILLING', 'EXTERNAL_RELUCTANT']

for pattern in patterns_of_interest:
    vol_rate = vol_patterns.get(pattern, 0) / vol_patterns.sum() * 100 if vol_patterns.sum() > 0 else 0
    std_rate = std_patterns.get(pattern, 0) / std_patterns.sum() * 100 if std_patterns.sum() > 0 else 0
    
    print(f"\n   {pattern}:")
    print(f"      Volunteer: {vol_rate:.1f}%")
    print(f"      Standard:  {std_rate:.1f}%")
    print(f"      Δ: {vol_rate - std_rate:+.1f}%")

# 3. Interpretation
print("\n\n3. INTERPRETATION:")
print("-" * 50)
print("   Expected Volunteer Soul Pattern:")
print("     - Higher SELF agency (they chose to come, they choose to return)")
print("     - May have RELUCTANT willingness (duty over desire)")
print()
print("   Expected Standard Path Pattern:")
print("     - Higher EXTERNAL agency (not their choice)")
print("     - Higher WILLING when EXTERNAL (accepting of journey interruption)")

---

# PART V: Transformation Profiles

**Objective**: Compare transformation patterns (fear, spirituality) across soul path categories.

**Test**: Use NEW before/after fields to measure transformation magnitude.

In [ ]:
# =============================================================================
# PART V: TRANSFORMATION PROFILES BY CATEGORY
# =============================================================================

print("=" * 70)
print("TRANSFORMATION PROFILES BY SOUL PATH CATEGORY")
print("=" * 70)
print("✓ NEW: Before/after fields enable transformation magnitude calculation")
print()

# Scale mapping
scale = {'NONE': 1, 'LOW': 2, 'MODERATE': 3, 'HIGH': 4, 'EXTREME': 5}

def compute_transformation(df, before_col, after_col, direction='decrease'):
    """Compute transformation magnitude for a before/after pair."""
    results = []
    for _, row in df.iterrows():
        before = row.get(before_col)
        after = row.get(after_col)
        if before in scale and after in scale:
            if direction == 'decrease':
                change = scale[before] - scale[after]  # Positive = decreased
            else:
                change = scale[after] - scale[before]  # Positive = increased
            results.append(change)
    return results

# 1. Death Fear Transformation by Volunteer Status
print("1. Death Fear Transformation:")
print("-" * 50)

vol_df = feature_df[feature_df['is_volunteer'] == 1]
std_df = feature_df[feature_df['is_volunteer'] == 0]

vol_fear_change = compute_transformation(vol_df, 'death_fear_before', 'death_fear_after', 'decrease')
std_fear_change = compute_transformation(std_df, 'death_fear_before', 'death_fear_after', 'decrease')

if vol_fear_change:
    print(f"   Volunteer (n={len(vol_fear_change)}): Mean change = {np.mean(vol_fear_change):+.2f}")
if std_fear_change:
    print(f"   Standard (n={len(std_fear_change)}): Mean change = {np.mean(std_fear_change):+.2f}")

# Statistical comparison
if len(vol_fear_change) >= 5 and len(std_fear_change) >= 5:
    stat, p_value = mannwhitneyu(vol_fear_change, std_fear_change, alternative='two-sided')
    print(f"\n   Mann-Whitney U: {stat:.2f}, p={p_value:.4f}")
    print(f"   {'✓ SIGNIFICANT' if p_value < 0.05 else '○ Not significant'} difference")

# 2. Spirituality Transformation
print("\n\n2. Spirituality Transformation:")
print("-" * 50)

vol_spirit_change = compute_transformation(vol_df, 'spirituality_before', 'spirituality_after', 'increase')
std_spirit_change = compute_transformation(std_df, 'spirituality_before', 'spirituality_after', 'increase')

if vol_spirit_change:
    print(f"   Volunteer (n={len(vol_spirit_change)}): Mean change = {np.mean(vol_spirit_change):+.2f}")
if std_spirit_change:
    print(f"   Standard (n={len(std_spirit_change)}): Mean change = {np.mean(std_spirit_change):+.2f}")

# 3. Interpretation
print("\n\n3. INTERPRETATION:")
print("-" * 50)
print("   Hypothesis: Volunteer Souls may show:")
print("     - Less fear reduction (already low baseline fear)")
print("     - Less spirituality increase (already high baseline)")
print("   Or alternatively:")
print("     - Greater transformation (mission confirmation effect)")

In [ ]:
# =============================================================================
# VISUALIZATION: Soul Path Classification
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Soul Path Classification Analysis', fontsize=14, fontweight='bold')

# 1. Volunteer Score Distribution
ax1 = axes[0, 0]
score_counts.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.axvline(x=VOLUNTEER_THRESHOLD - 0.5, color='red', linestyle='--', label=f'Threshold (≥{VOLUNTEER_THRESHOLD})')
ax1.set_title('Volunteer Score Distribution (NEW Split Fields)')
ax1.set_xlabel('Score (0-3)')
ax1.set_ylabel('Count')
ax1.legend()

# 2. Experience Valence (Threefold Path Proxy)
ax2 = axes[0, 1]
if len(path_counts) > 0:
    colors = {'AFFIRMATION': 'gold', 'NEGATION': 'darkred', 'TRANSITIONAL': 'gray', 'UNKNOWN': 'lightgray'}
    path_colors = [colors.get(p, 'steelblue') for p in path_counts.index]
    path_counts.plot(kind='bar', ax=ax2, color=path_colors, edgecolor='black')
    ax2.set_title('Threefold Path Estimate')
    ax2.set_xlabel('Path')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=45)

# 3. Feature Importance (if available)
ax3 = axes[1, 0]
if importance_df is not None and len(importance_df) > 0:
    top_features = importance_df.head(8)
    ax3.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue', edgecolor='black')
    ax3.set_yticks(range(len(top_features)))
    ax3.set_yticklabels([f.replace('_num', '') for f in top_features['Feature']])
    ax3.set_xlabel('Importance')
    ax3.set_title('Top Discriminant Features (RF)')
    ax3.invert_yaxis()
else:
    ax3.text(0.5, 0.5, 'ML analysis not performed', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Feature Importance')

# 4. Return Pattern by Volunteer Status
ax4 = axes[1, 1]
return_compare = pd.DataFrame({
    'Volunteer': vol_patterns.head(4) / vol_patterns.sum() * 100 if len(vol_patterns) > 0 else [],
    'Standard': std_patterns.head(4) / std_patterns.sum() * 100 if len(std_patterns) > 0 else []
})
if len(return_compare) > 0:
    return_compare.plot(kind='bar', ax=ax4, edgecolor='black')
    ax4.set_title('Return Patterns by Soul Type (NEW Schema)')
    ax4.set_xlabel('Pattern (Agency_Willingness)')
    ax4.set_ylabel('Percentage')
    ax4.tick_params(axis='x', rotation=45)
    ax4.legend(title='Type')
else:
    ax4.text(0.5, 0.5, 'No return pattern data', ha='center', va='center', transform=ax4.transAxes)

plt.tight_layout()
plt.savefig('../output/03_soul_path_classification.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: ../output/03_soul_path_classification.png")

---

# Conclusion: Soul Path Classification Summary

## Classification Results

| Category | Identification Method | Count | Percentage |
|----------|----------------------|-------|------------|
| **Volunteer Souls** | Score ≥ 2 on choice indicators | *see Part I* | *%* |
| **Path of Affirmation** | Positive valence NDEs | *see Part II* | *%* |
| **Path of Negation** | Distressing NDEs | *see Part II* | *%* |

## Schema Improvements Utilized

1. **Choice Field Split**: `chose_parents` + `chose_mission` + `chose_life_circumstances` (NEW)
2. **Return Split**: `return_agency` × `return_willingness` combinations (NEW)
3. **Before/After**: Transformation magnitude calculation (NEW)

## Discriminant Analysis Results

| Model | Accuracy | Key Features |
|-------|----------|--------------|
| **LDA** | *see Part III* | Linear separation |
| **Random Forest** | *see Part III* | Top features listed |

## Key Findings

1. **Volunteer Soul Indicators**: Pre-birth choice patterns form coherent cluster
2. **Return Patterns**: Agency × Willingness distinguishes soul types
3. **Transformation Profiles**: Different magnitudes by category

---

**Notebook**: `03_soul_path_classification.ipynb`  
**Schema Version**: 2026-01-06  
**Source Notebooks**: `volunteer_discriminant_analysis.ipynb`, `threefold_path_validation.ipynb`